# Week 1 — LLM Fundamentals & Environment Setup (Local with LiteLLM & Ollama)

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-01-llm-fundamentals-content.html`. Run the cells in order during lab time.

**You will practice:**
1. Making your first LLM call locally through LiteLLM and Ollama (`qwen2.5:14b`).
2. Seeing `temperature`, `top_p`, and `top_k` change model output on the *same* prompt.
3. Counting tokens for real and comparing it to the "1 token ≈ 4 characters" rule of thumb.
4. Making the *same* call through three tools: LiteLLM, a minimal Google ADK agent, and a minimal LangChain chat model.
5. Two open exercises.

**Environment:** Running 100% locally with Ollama, RTX GPU acceleration, and LiteLLM.


In [12]:
%pip install -q --upgrade litellm google-adk langchain-community langchain-ollama python-dotenv


Note: you may need to restart the kernel to use updated packages.


## 1. Environment check

Make one call to confirm LiteLLM is communicating with your local Ollama model.

In [2]:
from dotenv import load_dotenv
from litellm import completion

load_dotenv()
MODEL = "ollama_chat/qwen2.5:14b"

response = completion(
    model=MODEL,
    messages=[{"role": "user", "content": "In one sentence, what is a large language model?"}],
)
print(response.choices[0].message.content)


A large language model is a sophisticated artificial intelligence system designed to process and generate human-like text based on a vast amount of learned data from the internet and other sources.


## 2. Sampling parameters: temperature, top-p, top-k

We'll send the **same prompt** several times, changing only `temperature`. Watch how the output goes from
predictable to varied.

In [5]:
PROMPT = "Give me one creative name for a coffee shop. Reply with just the name."

for temp in [0.0, 0.4, 0.9, 1.5]:
    response = completion(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT}],
        temperature=temp,
        max_tokens=20,
    )
    print(f"temperature={temp:<4} -> {response.choices[0].message.content.strip()}")


temperature=0.0  -> Mocha Muse
temperature=0.4  -> Mocha Muse
temperature=0.9  -> Brewtopia
temperature=1.5  -> Sippery Slope咖啡馆


Run the cell above a few times. At `temperature=0.0` the answer should barely change between runs.
At `temperature=1.5` you should see real variety (and occasionally something a little unhinged — that's expected).

Now let's isolate `top_p` and `top_k` by holding temperature fixed at a mid-range value.

In [26]:
for top_p in [0.1, 0.5, 0.95]:
    response = completion(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT}],
        temperature=1.5,
        top_p=top_p,
        max_tokens=20,
    )
    print(f"top_p={top_p:<5} -> {response.choices[0].message.content.strip()}")

print()
for top_k in [1, 5, 40]:
    response = completion(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT}],
        temperature=1.5,
        top_k=top_k,
        max_tokens=20,
    )
    print(f"top_k={top_k:<3} -> {response.choices[0].message.content.strip()}")


top_p=0.1   -> Mocha Muse
top_p=0.5   -> Brewtopia
top_p=0.95  -> Bean Voyage

top_k=1   -> Mocha Muse
top_k=5   -> Bean Voyage
top_k=40  -> Sip&Soul


## 3. Tokens: count them for real

The rule of thumb is "1 token ≈ 4 characters ≈ 0.75 words" for English. Let's check it against the real
tokenizer using `litellm.token_counter`.

In [17]:
import litellm

samples = [
    "Hi!",
    "The capital of France is Paris.",
    "AI Agentic Engineering is a course about building autonomous systems with large language models, "
    "retrieval-augmented generation, and multi-agent orchestration frameworks like Google ADK and LangGraph.",
]

for text in samples:
    tokens = litellm.token_counter(model=MODEL, text=text)
    chars = len(text)
    ratio = chars / tokens if tokens else 0
    print(f"chars={chars:<4} tokens={tokens:<4} chars/token={ratio:.2f}  | {text[:50]!r}")


chars=3    tokens=2    chars/token=1.50  | 'Hi!'
chars=31   tokens=7    chars/token=4.43  | 'The capital of France is Paris.'
chars=200  tokens=36   chars/token=5.56  | 'AI Agentic Engineering is a course about building '


## 3b. Activity 3 — Prompt & Parameter Detective (solo version)

The in-class version has the instructor hand each group three outputs from undisclosed
`temperature`/`top_p` settings. Doing it solo with your local model: generate the three outputs
yourself, but shuffle and hide the labels so you're analyzing them blind, same as in class.

Run the next cell, then **before printing `_answer_key`**:
1. Rank `Output A/B/C` from most to least deterministic-looking.
2. Justify each ranking in one sentence using: sampling, distribution, nucleus, peaked/flat.
3. Say which setting you'd ship for: (a) a support-ticket categorizer, (b) a creative story
   generator, (c) a JSON-producing data-extraction endpoint.
4. Only then reveal `_answer_key` and check how close your ranking was.


In [7]:
import random

PROMPT_DETECTIVE = "Write a one-sentence opening for a short story."

SETTINGS = [
    {"label": "low",  "temperature": 0.0, "top_p": 0.1},
    {"label": "mid",  "temperature": 0.7, "top_p": 0.9},
    {"label": "high", "temperature": 1.4, "top_p": 1.0},
]

shuffled = SETTINGS.copy()
random.shuffle(shuffled)

outputs = []
for cfg in shuffled:
    response = completion(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT_DETECTIVE}],
        temperature=cfg["temperature"],
        top_p=cfg["top_p"],
        max_tokens=60,
    )
    outputs.append(response.choices[0].message.content.strip())

for i, text in enumerate(outputs):
    print(f"Output {chr(65+i)}:\n{text}\n")

_answer_key = {chr(65+i): shuffled[i] for i in range(len(shuffled))}


Output A:
On the eve of her thirtieth birthday, Eliza discovered an old, unmarked key hidden within the pages of a childhood diary, setting into motion a series of events that would unravel the mysteries of her family's past.

Output B:
On the dusty edge of Mapledown, where the summer air tasted of honeysuckle and forgotten dreams, Evelyn found a curious, old map tucked inside the sole remaining copy of the town's long-lost archive, whispering secrets of a world hidden just beyond the boundaries of the ordinary.

Output C:
On the cold, fog-laden morning of her thirtieth birthday, Evelyn discovered an old, unmarked envelope slipped between the pages of the dusty family Bible, its contents threatening to unravel the carefully constructed narrative of her past.



In [8]:
# Reveal — run this only after you've written your ranking and justification above.

print("True settings, from most to least deterministic:")
for label in sorted(_answer_key, key=lambda l: (_answer_key[l]["temperature"], _answer_key[l]["top_p"])):
    cfg = _answer_key[label]
    print(f"  Output {label}: temperature={cfg['temperature']}, top_p={cfg['top_p']}  ({cfg['label']})")


True settings, from most to least deterministic:
  Output A: temperature=0.0, top_p=0.1  (low)
  Output C: temperature=0.7, top_p=0.9  (mid)
  Output B: temperature=1.4, top_p=1.0  (high)


## 4. Same call, three tools

Below, the identical prompt is sent through: **(A)** LiteLLM, **(B)** a one-line **Google ADK**
agent run with `LiteLlm`, and **(C)** a **LangChain** chat model (`ChatLiteLLM`).

In [18]:
# --- A. LiteLLM ---
prompt = "Name one famous mathematician and the theorem they're best known for."

response = completion(model=MODEL, messages=[{"role": "user", "content": prompt}])
print("A) LiteLLM:\n", response.choices[0].message.content)


A) LiteLLM:
 One famous mathematician is Pythagoras, and he is best known for the Pythagorean theorem. This theorem states that in a right-angled triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides. This can be written as: \(a^2 + b^2 = c^2\), where \(c\) represents the length of the hypotenuse, and \(a\) and \(b\) represent the lengths of the other two sides.


In [ ]:
# --- B. Google ADK (minimal agent with LiteLlm) ---
import asyncio
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

adk_agent = Agent(
    model=LiteLlm(model=MODEL),
    name="week1_demo_agent",
    instruction="Answer concisely, in 1-2 sentences.",
)

async def ask_adk_agent(agent, prompt, app_name="week1_app", user_id="student"):
    """Small reusable helper: runs one prompt through an ADK agent and returns the final text."""
    session_service = InMemorySessionService()
    runner = Runner(
        agent=agent,
        app_name=app_name,
        session_service=session_service,
        auto_create_session=True,
    )
    session = await session_service.create_session(app_name=app_name, user_id=user_id)
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = ""
    async for event in runner.run_async(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text:
                    final_text += p.text
    return final_text

# En Jupyter Notebook usamos 'await' directamente:
adk_response = await ask_adk_agent(adk_agent, prompt)
print("B) Google ADK agent:\n", adk_response)


B) Google ADK agent:
 Pythagoras is famous for the Pythagorean theorem, which states that in a right-angled triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides.


In [20]:
# --- C. LangChain chat model ---
from langchain_ollama import ChatOllama

ollama_model_name = MODEL.replace("ollama_chat/", "").replace("ollama/", "")
llm = ChatOllama(model=ollama_model_name)
lc_response = llm.invoke(prompt)
print("C) LangChain:\n", lc_response.content)


C) LangChain:
 One famous mathematician is Pythagoras, and he is best known for the Pythagorean theorem. This theorem states that in a right-angled triangle, the square of the length of the hypotenuse (the side opposite the right angle) is equal to the sum of the squares of the lengths of the other two sides. This can be expressed algebraically as \(a^2 + b^2 = c^2\), where \(c\) represents the length of the hypotenuse, and \(a\) and \(b\) represent the lengths of the other two sides.


## 5. Exercises

Complete both before the peer code review activity.

In [38]:
# TODO Exercise 1 — Your own parameter sweep
# Pick a prompt of your own (something with a clearly "creative" vs "factual" framing).
# Sweep temperature over at least 4 values and print the results, like section 2 above.

MY_PROMPT = ("Imagine a world where magic exists, but it must obey the laws of physics. "
            "Create a magical item for that world and describe how it works. "
            "The magic can produce seemingly supernatural effects, but it cannot "
            "violate conservation of energy, conservation of momentum, causality, "
            "or the universal speed limit of light. "
            "Respond in a single concise paragraph of no more than 100 words.")

for temp in [0.0, 0.4, 0.9, 1.5]:
    res = completion(
        model=MODEL,
        messages=[{"role": "user", "content": MY_PROMPT}],
        temperature=temp,
    )
    print(f"temperature={temp:<4} -> {res.choices[0].message.content.strip()}")


temperature=0.0  -> In this world, the Amulet of Levitation is a magical item that allows the wearer to hover and move in any direction, but it adheres strictly to physical laws. The amulet generates a localized gravitational field that can be adjusted to counteract Earth's gravity, enabling levitation. To move horizontally, the amulet subtly shifts the direction of the gravitational field, creating an apparent force that propels the wearer. Energy for these effects is drawn from the wearer's body, ensuring that the total energy is conserved, and the amulet's magic respects the speed of light and causality, preventing instantaneous movement or teleportation.
temperature=0.4  -> In this world, the Amulet of Lumina is a magical pendant that harnesses the principle of energy conservation by converting ambient thermal energy into magical energy. When worn, it absorbs heat from the surroundings and converts it into a form of magical energy that can be channeled by the wearer. This energy ca

In [22]:
# TODO Exercise 2 — Estimate tokens without calling the API
# Write a function `estimate_tokens(text)` that approximates token count using ONLY the
# "1 token ≈ 4 characters" rule (no API call). Then compare it against the real token_counter()
# result for the three `samples` from section 3, and print the % error for each.

def estimate_tokens(text: str) -> float:
    return len(text) / 4.0

for text in samples:
    real = litellm.token_counter(model=MODEL, text=text)
    estimate = estimate_tokens(text)
    error = abs(estimate - real) / real * 100 if real else 0
    print(f"Real: {real:<3} | Estimado: {estimate:<4.1f} | Error: {error:<5.1f}% | {text[:45]!r}")


Real: 2   | Estimado: 0.8  | Error: 62.5 % | 'Hi!'
Real: 7   | Estimado: 7.8  | Error: 10.7 % | 'The capital of France is Paris.'
Real: 36  | Estimado: 50.0 | Error: 38.9 % | 'AI Agentic Engineering is a course about buil'


## Next week

Week 2 — **Context Engineering I**: system prompts, few-shot prompting, chain-of-thought, and forcing structured
JSON output. See `week-02-context-engineering-i-content.html`.